# Activation Distribution Statistics

Statistical analysis of activation distributions at each layer — moments, heavy tails, normality, sparsity patterns, and multimodality. Understanding these distributions is key to quantization, pruning, and identifying anomalous computation.

**References:**
- Dettmers et al. (2022) "LLM.int8(): 8-bit Matrix Multiplication"
- Sun et al. (2024) "Massive Activations in Large Language Models"

## Why This Matters

Activation statistics — moments, kurtosis, normality, sparsity, and multimodality — provide a quantitative profile of how each layer processes information. Unusual statistics (e.g., extreme kurtosis or bimodal distributions) flag components worth investigating in detail.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.activation_statistics import (
    layer_activation_moments,
    kurtosis_profile,
    normality_test_by_layer,
    activation_sparsity_pattern,
    multimodality_detection,
)

# Create a small model
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens = jnp.array([0, 5, 10, 15, 20, 25, 30, 35, 40, 45])
print(f"Model: {cfg.n_layers} layers, d_model={cfg.d_model}")
print(f"Input: {len(tokens)} tokens")

## 1. Layer Activation Moments

Compute mean, variance, skewness, and excess kurtosis of activations at each layer. These statistics reveal the shape of activation distributions — Gaussian vs heavy-tailed, symmetric vs skewed.

In [ ]:
moments = layer_activation_moments(model, tokens)

for l in range(cfg.n_layers):
    print(f"Layer {l}: mean={moments['means'][l]:.4f}, "
          f"var={moments['variances'][l]:.4f}, "
          f"skew={moments['skewnesses'][l]:.4f}, "
          f"kurtosis={moments['kurtoses'][l]:.4f}")

print(f"\nKurtosis > 0 means heavier tails than Gaussian (leptokurtic)")
print(f"Kurtosis < 0 means lighter tails than Gaussian (platykurtic)")

## 2. Kurtosis Profile

Track heavy-tailedness through the network. High kurtosis indicates outlier dimensions (important for quantization). Also available per-dimension for fine-grained analysis.

In [ ]:
kurt = kurtosis_profile(model, tokens, per_dimension=True)

print(f"Layer kurtosis: {kurt['layer_kurtosis']}")
print(f"Max kurtosis at layer: {kurt['max_kurtosis_layer']}")
print(f"Kurtosis trend: {kurt['kurtosis_trend']}")
print(f"\nPer-dimension kurtosis shape: {kurt['per_dim_kurtosis'].shape}")
print(f"Dimensions with kurtosis > 1 (outlier-prone):")
for l in range(cfg.n_layers):
    n_outlier = int(np.sum(kurt['per_dim_kurtosis'][l] > 1))
    print(f"  Layer {l}: {n_outlier}/{cfg.d_model} dimensions")

## 3. Normality Testing

The Jarque-Bera test measures how far activations deviate from a Gaussian distribution. This matters for quantization (Gaussian assumptions) and understanding computation type.

In [ ]:
normality = normality_test_by_layer(model, tokens)

for l in range(cfg.n_layers):
    print(f"Layer {l}: JB={normality['jb_statistics'][l]:.2f}, "
          f"normality={normality['normality_scores'][l]:.4f}")

print(f"\nMost Gaussian layer: {normality['most_gaussian_layer']}")
print(f"Least Gaussian layer: {normality['least_gaussian_layer']}")
print(f"\nNormality score: 1.0 = perfectly Gaussian, 0.0 = very non-Gaussian")

## 4. Activation Sparsity Patterns

Measure how sparse activations are at each layer. Sparsity reveals which layers use focused vs distributed computation, and identifies candidates for pruning.

In [ ]:
sparsity = activation_sparsity_pattern(model, tokens, threshold=0.01)

for l in range(cfg.n_layers):
    print(f"Layer {l}: sparsity={sparsity['sparsity_ratios'][l]:.4f}, "
          f"mean_mag={sparsity['mean_magnitudes'][l]:.4f}, "
          f"L1={sparsity['l1_norms'][l]:.2f}")

print(f"\nSparsest layer: {sparsity['sparsest_layer']}")
print(f"Densest layer: {sparsity['densest_layer']}")

## 5. Multimodality Detection

Detect whether activations have multiple modes (clusters). Multimodal distributions suggest the layer computes discrete features rather than continuous representations.

In [ ]:
modes = multimodality_detection(model, tokens, n_bins=20)

for l in range(cfg.n_layers):
    print(f"Layer {l}: {modes['n_modes_per_layer'][l]} mode(s), "
          f"bimodality_coeff={modes['bimodality_coefficients'][l]:.4f}")

print(f"\nMost multimodal layer: {modes['most_multimodal_layer']}")
print(f"\nBimodality coefficient > 0.555 suggests bimodal distribution")
print(f"\nHistogram shapes available in modes['layer_histograms']")